# Chapter 19 — Empirical Project: Returns to Education
## University of Neuchâtel — Python for Econometrics

**Research Question:** What is the return to education on wages?

This chapter applies all techniques learned in the course:
1. Data exploration and EDA
2. Multiple OLS models (build-up strategy)
3. Ability bias — adding IQ as proxy
4. Nonlinearity — quadratic experience
5. Interaction effects — race × education
6. Robustness: heteroscedasticity, RESET, robust SEs
7. Publication-quality coefficient plot

**Format:** One cell per step. Manual first, statsmodels second, `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import statsmodels.stats.diagnostic as diag
from scipy import stats
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Data

Dataset: `wage2` — sample of US men aged 18-28 (1980 NLSY survey)

Key variables:
- `lwage`: log monthly wage (outcome)
- `educ`: years of schooling
- `IQ`: IQ score (proxy for ability)
- `exper`: years of work experience
- `married`: 1=married
- `black`: 1=Black
- `south`: 1=lives in South
- `urban`: 1=lives in urban area

In [ ]:
df = woo.dataWoo('wage2')
print('Shape:', df.shape)
print(df[['lwage','educ','IQ','exper','married','black','south','urban']].head(10))

In [ ]:
print('Missing values per column:')
print(df[['lwage','educ','IQ','exper','married','black','south','urban']].isnull().sum())

In [ ]:
df_clean = df.dropna(subset=['lwage','educ','IQ','exper','married','black','south','urban'])
print(f'Clean sample: {len(df_clean)} observations')
print(df_clean[['lwage','educ','IQ','exper']].describe().round(3))

## 2 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(11,8))
axes[0,0].hist(np.exp(df_clean['lwage']), bins=25, edgecolor='black')
axes[0,0].set_title('Wage distribution (levels)')
axes[0,0].set_xlabel('Monthly wage ($)')
axes[0,1].hist(df_clean['lwage'], bins=25, edgecolor='black', color='orange')
axes[0,1].set_title('log(Wage) distribution — approx. normal')
axes[0,1].set_xlabel('log wage')
axes[1,0].scatter(df_clean['educ'], df_clean['lwage'], alpha=0.3, s=15)
axes[1,0].set_xlabel('Education')
axes[1,0].set_ylabel('log wage')
axes[1,0].set_title('Education vs log wage')
axes[1,1].scatter(df_clean['IQ'], df_clean['lwage'], alpha=0.3, s=15, color='green')
axes[1,1].set_xlabel('IQ')
axes[1,1].set_ylabel('log wage')
axes[1,1].set_title('IQ vs log wage')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch19_01_eda.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Correlation matrix:')
corr = df_clean[['lwage','educ','IQ','exper']].corr().round(3)
print(corr)

## 3 — Hypotheses

1. **H1:** Return to education is positive ($\beta_{educ} > 0$)
2. **H2:** Controlling for IQ (ability) reduces the education coefficient (ability bias)
3. **H3:** Returns to experience are nonlinear (quadratic)
4. **H4:** Returns to education may differ by race (interaction)

## 4 — Model 1: Simple OLS (lwage ~ educ)

In [ ]:
n = len(df_clean)
x = df_clean['educ'].values
y = df_clean['lwage'].values
b1_m1 = np.sum((x-x.mean())*(y-y.mean())) / np.sum((x-x.mean())**2)
b0_m1 = y.mean() - b1_m1 * x.mean()
print(f'Model 1 (manual): lwage = {b0_m1:.4f} + {b1_m1:.4f}*educ')

In [ ]:
res1 = smf.ols('lwage ~ educ', data=df_clean).fit()
print(f'Model 1: educ coeff = {res1.params["educ"]:.4f}')
print(f'Match: {np.isclose(b1_m1, res1.params["educ"])}')
print(f'Interpretation: 1 yr education → {100*res1.params["educ"]:.1f}% higher wage')

## 5 — Model 2: Add Controls

In [ ]:
n2 = len(df_clean)
X2 = np.column_stack([
    np.ones(n2),
    df_clean['educ'].values,
    df_clean['exper'].values,
    df_clean['married'].values,
    df_clean['south'].values,
    df_clean['urban'].values,
    df_clean['black'].values
])
y2 = df_clean['lwage'].values
beta2 = np.linalg.solve(X2.T @ X2, X2.T @ y2)
labels2 = ['const','educ','exper','married','south','urban','black']
print('Model 2 (manual):')
for lbl, b in zip(labels2, beta2):
    print(f'  {lbl:10s}: {b:.4f}')

In [ ]:
res2 = smf.ols('lwage ~ educ + exper + married + south + urban + black', data=df_clean).fit()
sm2 = res2.params
print('Model 2 comparison:')
for lbl, b_man in zip(labels2, beta2):
    match = np.isclose(b_man, sm2[lbl])
    print(f'  {lbl:10s}: manual={b_man:.4f}, sm={sm2[lbl]:.4f}, match={match}')

## 6 — Model 3: Add IQ (Proxy for Ability)

In [ ]:
X3 = np.column_stack([X2, df_clean['IQ'].values])
beta3 = np.linalg.solve(X3.T @ X3, X3.T @ y2)
print(f'Model 3 educ coeff (manual): {beta3[1]:.4f}')

In [ ]:
res3 = smf.ols('lwage ~ educ + IQ + exper + married + south + urban + black', data=df_clean).fit()
print(f'Model 3 educ coeff (sm): {res3.params["educ"]:.4f}')
print(f'Match: {np.isclose(beta3[1], res3.params["educ"])}')
bias = res2.params['educ'] - res3.params['educ']
print(f'\nAbility bias = Model2.educ - Model3.educ = {bias:.4f}')
print('Adding IQ reduces educ coefficient → confirms upward ability bias in Model 2')

## 7 — Model 4: Quadratic Experience

In [ ]:
df_clean = df_clean.copy()
df_clean['exper2'] = df_clean['exper']**2
print('exper2 added (first 5 rows):')
print(df_clean[['exper','exper2']].head())

In [ ]:
X4 = np.column_stack([X3, df_clean['exper2'].values])
beta4 = np.linalg.solve(X4.T @ X4, X4.T @ y2)
print(f'exper coeff (manual):  {beta4[2]:.4f}')
print(f'exper2 coeff (manual): {beta4[-1]:.4f}')

In [ ]:
res4 = smf.ols('lwage ~ educ + IQ + exper + exper2 + married + south + urban + black', data=df_clean).fit()
b_exp  = res4.params['exper']
b_exp2 = res4.params['exper2']
print(f'exper:  {b_exp:.4f}')
print(f'exper2: {b_exp2:.4f}')
print(f'Match exper:  {np.isclose(beta4[2], b_exp)}')
print(f'Match exper2: {np.isclose(beta4[-1], b_exp2)}')
turning_pt = -b_exp / (2 * b_exp2)
print(f'Turning point (max wage): {turning_pt:.1f} years of experience')

## 8 — Model 5: Race × Education Interaction

In [ ]:
df_clean['black_educ'] = df_clean['black'] * df_clean['educ']
print('Interaction term black_educ (first 5 rows):')
print(df_clean[['black','educ','black_educ']].head())

In [ ]:
res5 = smf.ols('lwage ~ educ + IQ + exper + exper2 + married + south + urban + black + black_educ',
               data=df_clean).fit()
b_educ  = res5.params['educ']
d_black = res5.params['black_educ']
print(f'Return to educ (non-Black): {b_educ:.4f}  ({100*b_educ:.1f}%/yr)')
print(f'Return to educ (Black):     {b_educ+d_black:.4f}  ({100*(b_educ+d_black):.1f}%/yr)')
print(f'Differential delta_1:       {d_black:.4f}  (p={res5.pvalues["black_educ"]:.4f})')

## 9 — Robustness Checks

In [ ]:
# Heteroscedasticity: Breusch-Pagan
e5 = res5.resid.values
X5 = res5.model.exog
bp = diag.het_breuschpagan(e5, X5[:,1:])
print(f'Breusch-Pagan: LM={bp[0]:.4f}, p={bp[1]:.4f}')
if bp[1] < 0.05:
    print('  Heteroscedasticity detected → use robust SEs')

In [ ]:
# RESET test
yhat = res5.fittedvalues
df_clean['yhat2'] = yhat**2
df_clean['yhat3'] = yhat**3
res5_aug = smf.ols('lwage ~ educ+IQ+exper+exper2+married+south+urban+black+black_educ+yhat2+yhat3',
                   data=df_clean).fit()
f_reset = res5_aug.f_test('yhat2=0, yhat3=0')
print(f'RESET test: F={f_reset.fvalue[0][0]:.4f}, p={f_reset.pvalue:.4f}')
if f_reset.pvalue > 0.05:
    print('  Model passes RESET: functional form ok')

In [ ]:
# Robust SEs
res5_hc1 = res5.get_robustcov_results(cov_type='HC1')
print('Comparison: OLS vs HC1 SE for educ:')
print(f'  OLS SE: {res5.bse["educ"]:.4f}')
print(f'  HC1 SE: {res5_hc1.bse["educ"]:.4f}')

## 10 — Model Comparison Table

In [ ]:
models = [res1, res2, res3, res4, res5]
model_names = ['M1 Simple','M2 Controls','M3 +IQ','M4 +exper2','M5 +interaction']
sumtab = pd.DataFrame({
    'Model': model_names,
    'b_educ': [m.params.get('educ', np.nan) for m in models],
    'b_IQ':   [m.params.get('IQ', np.nan) for m in models],
    'R2':     [m.rsquared for m in models],
    'N':      [int(m.nobs) for m in models]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch19_model_comparison.csv'), index=False)
print('Saved table_ch19_model_comparison.csv')

## 11 — Publication-Quality Coefficient Plot

In [ ]:
key_vars = ['educ','IQ','exper','married','south','urban','black','black_educ']
coefs = [res5.params[v] for v in key_vars]
ci_lo = [res5.conf_int().loc[v,0] for v in key_vars]
ci_hi = [res5.conf_int().loc[v,1] for v in key_vars]
err_lo = [coefs[i]-ci_lo[i] for i in range(len(key_vars))]
err_hi = [ci_hi[i]-coefs[i] for i in range(len(key_vars))]
fig, ax = plt.subplots(figsize=(8,5))
y_pos = range(len(key_vars))
ax.errorbar(coefs, y_pos, xerr=[err_lo, err_hi], fmt='o', capsize=5, linewidth=2)
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(key_vars)
ax.set_xlabel('Coefficient estimate')
ax.set_title('Model 5: Coefficients with 95% CI')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch19_02_coeff_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12 — Conclusion

**Main findings:**
1. Return to education ≈ 6-8% per year of schooling (log-level model)
2. Ability bias is real: adding IQ reduces the education coefficient
3. Experience has a concave effect (positive but diminishing)
4. The education-race differential is modest and often insignificant

**Limitations:**
- IQ is an imperfect proxy for ability
- Education may still be endogenous (IV/2SLS would be more rigorous)
- Sample limited to men aged 18-28 in 1980

**Extensions:**
- Use `card` dataset with college proximity IV
- Panel data (wagepan) with fixed effects to remove unobserved heterogeneity